In [1]:
import re
import math
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

if torch.backends.mps.is_available():
    device = torch.device("mps")
    device_name = "mps"
else:
    device = torch.device("cpu")
    device_name = "cpu"

print(f"Using device: {device_name}")
torch.set_grad_enabled(False)

Using device: mps


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [2]:
model_name = "textattack/roberta-base-MRPC"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Model labels: {model.config.id2label}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: textattack/roberta-base-MRPC
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: textattack/roberta-base-MRPC
Model labels: {0: 'LABEL_0', 1: 'LABEL_1'}


In [3]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Dataset split: glue/mrpc validation")
print(f"Number of examples: {len(dataset)}")
print("Example row:")
print(dataset[0])

Dataset split: glue/mrpc validation
Number of examples: 408
Example row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
token_pattern = re.compile(r"\b\w+\b")

def tokenize_words(text):
    return set(token_pattern.findall(text.lower()))

def lexical_overlap(sentence1, sentence2):
    a = tokenize_words(sentence1)
    b = tokenize_words(sentence2)
    if not a and not b:
        return 1.0
    union = a | b
    if not union:
        return 0.0
    return len(a & b) / len(union)

overlaps = [lexical_overlap(row["sentence1"], row["sentence2"]) for row in dataset]
labels_all = dataset["label"]

negative_overlaps = [o for o, y in zip(overlaps, labels_all) if y == 0]
positive_overlaps = [o for o, y in zip(overlaps, labels_all) if y == 1]

high_overlap_negative_threshold = float(np.quantile(negative_overlaps, 0.80))
low_overlap_positive_threshold = float(np.quantile(positive_overlaps, 0.20))

hard_indices = [
    i for i, (o, y) in enumerate(zip(overlaps, labels_all))
    if (y == 0 and o >= high_overlap_negative_threshold) or (y == 1 and o <= low_overlap_positive_threshold)
]

hard_subset = dataset.select(hard_indices)
hard_overlaps = [overlaps[i] for i in hard_indices]

print("Constructed hard-case subset from validation data.")
print(f"High-overlap negative threshold (80th pct among negatives): {high_overlap_negative_threshold:.4f}")
print(f"Low-overlap positive threshold (20th pct among positives): {low_overlap_positive_threshold:.4f}")
print(f"Hard-case subset size: {len(hard_subset)}")
print("First hard-case example:")
print({**hard_subset[0], "lexical_overlap": hard_overlaps[0]})

Constructed hard-case subset from validation data.
High-overlap negative threshold (80th pct among negatives): 0.5556
Low-overlap positive threshold (20th pct among positives): 0.4333
Hard-case subset size: 86
First hard-case example:
{'sentence1': 'The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .', 'sentence2': 'The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .', 'label': 0, 'idx': 25, 'lexical_overlap': 0.56}


In [ ]:
batch_size = 32
all_logits = []
all_probabilities = []
all_predictions = []
all_confidences = []

for start in range(0, len(hard_subset), batch_size):
    batch = hard_subset[start:start + batch_size]
    encoded = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}

    outputs = model(**encoded)
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    confidences, predictions = torch.max(probabilities, dim=-1)

    all_logits.extend(logits.detach().cpu().tolist())
    all_probabilities.extend(probabilities.detach().cpu().tolist())
    all_predictions.extend(predictions.detach().cpu().tolist())
    all_confidences.extend(confidences.detach().cpu().tolist())

print(f"Completed manual model inference for {len(all_predictions)} hard-case examples.")
print("Sample logits/probabilities/prediction:")
print(all_logits[0])
print(all_probabilities[0])
print(all_predictions[0], all_confidences[0])

In [ ]:
subset_labels = hard_subset["label"]

accuracy = accuracy_score(subset_labels, all_predictions)
precision, recall, f1, _ = precision_recall_fscore_support(subset_labels, all_predictions, average="binary", zero_division=0)
cm = confusion_matrix(subset_labels, all_predictions)
avg_confidence = float(np.mean(all_confidences))

correct_overlaps = [o for o, y_true, y_pred in zip(hard_overlaps, subset_labels, all_predictions) if y_true == y_pred]
incorrect_overlaps = [o for o, y_true, y_pred in zip(hard_overlaps, subset_labels, all_predictions) if y_true != y_pred]

avg_overlap_correct = float(np.mean(correct_overlaps)) if correct_overlaps else float("nan")
avg_overlap_incorrect = float(np.mean(incorrect_overlaps)) if incorrect_overlaps else float("nan")

print("Subset evaluation metrics:")
print(f"Accuracy                    : {accuracy:.4f}")
print(f"Precision                   : {precision:.4f}")
print(f"Recall                      : {recall:.4f}")
print(f"F1                          : {f1:.4f}")
print(f"Average confidence          : {avg_confidence:.4f}")
print(f"Avg lexical overlap correct : {avg_overlap_correct:.4f}")
print(f"Avg lexical overlap wrong   : {avg_overlap_incorrect:.4f}")
print("Confusion matrix:")
print(cm)

In [ ]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}
errors = []

for local_idx, (true_label, pred_label, conf, overlap, probs, logits) in enumerate(
    zip(subset_labels, all_predictions, all_confidences, hard_overlaps, all_probabilities, all_logits)
):
    if true_label != pred_label:
        row = hard_subset[local_idx]
        errors.append({
            "subset_index": local_idx,
            "original_index": hard_indices[local_idx],
            "sentence1": row["sentence1"],
            "sentence2": row["sentence2"],
            "true_label": true_label,
            "pred_label": pred_label,
            "confidence": float(conf),
            "lexical_overlap": float(overlap),
            "probabilities": [float(x) for x in probs],
            "logits": [float(x) for x in logits],
        })

errors = sorted(errors, key=lambda x: (x["confidence"], x["lexical_overlap"]), reverse=True)
num_examples_to_show = min(10, len(errors))

print(f"Total hard-case errors: {len(errors)}")
print(f"Showing {num_examples_to_show} highest-confidence hard-case errors")

for example in errors[:num_examples_to_show]:
    print(f"Subset index: {example['subset_index']}")
    print(f"Original index: {example['original_index']}")
    print(f"sentence1: {example['sentence1']}")
    print(f"sentence2: {example['sentence2']}")
    print(f"true label: {example['true_label']} ({label_map[example['true_label']]})")
    print(f"pred label: {example['pred_label']} ({label_map[example['pred_label']]})")
    print(f"confidence: {example['confidence']:.4f}")
    print(f"lexical overlap: {example['lexical_overlap']:.4f}")
    print(f"probabilities: {example['probabilities']}")
    print(f"logits: {example['logits']}")
    print("-" * 80)

In [ ]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("subset_definition=hard_cases_from_dataset_itself")
print("hard_case_rule=(label=0 and lexical_overlap>=neg_80pct) OR (label=1 and lexical_overlap<=pos_20pct)")
print("inference_method=manual_tokenizer_model_forward_pass")
print("input_format=paired_sequence_tokenization_sentence1_sentence2")
print(f"device={device_name}")
print(f"num_examples_full_validation={len(dataset)}")
print(f"num_examples_subset={len(hard_subset)}")
print(f"high_overlap_negative_threshold={high_overlap_negative_threshold:.4f}")
print(f"low_overlap_positive_threshold={low_overlap_positive_threshold:.4f}")
print(f"accuracy={accuracy:.4f}")
print(f"precision={precision:.4f}")
print(f"recall={recall:.4f}")
print(f"f1={f1:.4f}")
print(f"average_confidence={avg_confidence:.4f}")
print(f"average_lexical_overlap_correct={avg_overlap_correct:.4f}")
print(f"average_lexical_overlap_incorrect={avg_overlap_incorrect:.4f}")
print(f"num_errors={len(errors)}")